In [1]:
# ============================================================
# pilot4_synthetic_step4_lr_5000
#
# Build synthetic Step4T-style implicit-transitive probe samples
# from current Step4T output format.
#
# Output:
#   pilot4_synthetic_step4_lr_5000.json
#   pilot4_synthetic_step4_lr_5000.zip
#
# This file can be used as Step5 input.
# ============================================================

import os
import re
import json
import uuid
import random
import shutil
import zipfile
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Dict, List, Tuple, Optional

from google.colab import files
from google.colab import drive

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

TARGET_NUM_SAMPLES = 5000

OUTPUT_BASENAME = "pilot4_synthetic_step4_lr_5000"
OUTPUT_JSON = f"{OUTPUT_BASENAME}.json"
OUTPUT_ZIP = f"{OUTPUT_BASENAME}.zip"
STEP5_COMPAT_JSON = f"step4T_{OUTPUT_BASENAME}_implicit_transitive_probe_samples.json"

WORK_DIR = Path("/content/pilot4_synthetic_step4_lr_5000")
INPUT_DIR = WORK_DIR / "input_step4"
OUTPUT_DIR = WORK_DIR / "output"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Scene split is designed to match your current Step6 setup:
#   train scenes = FloorPlan1-4
#   test scenes  = FloorPlan5-6
TRAIN_SCENES = ["FloorPlan1", "FloorPlan2", "FloorPlan3", "FloorPlan4"]
TEST_SCENES = ["FloorPlan5", "FloorPlan6"]
ALL_SCENES = TRAIN_SCENES + TEST_SCENES

# Use disjoint object pools for train and test scenes.
TRAIN_POOL_NAME = "train"
TEST_POOL_NAME = "test"

LABELS = ["left_of", "right_of"]

IMPLICIT_RULES = [
    "chain_same_direction",
    "shared_anchor_opposite_sides",
    "anchor_between_reversed_surface_form",
]

INVERSE_RELATION_MAP = {
    "left_of": "right_of",
    "right_of": "left_of",
}

print("Synthetic Step4T construction config loaded.")
print("TARGET_NUM_SAMPLES:", TARGET_NUM_SAMPLES)
print("Output:", OUTPUT_JSON)

Synthetic Step4T construction config loaded.
TARGET_NUM_SAMPLES: 5000
Output: pilot4_synthetic_step4_lr_5000.json


In [2]:
# ============================================================
# Upload current Step4T output
#
# Accepted:
#   - .zip containing step4T_*_implicit_transitive_probe_samples.json
#   - one or more .json files
#   - .jsonl files
# ============================================================

print("Upload current Step4T output files or zip.")
uploaded = files.upload()

if not uploaded:
    raise FileNotFoundError("No files uploaded.")

# Clear old input files.
for p in INPUT_DIR.glob("*"):
    if p.is_file():
        p.unlink()
    elif p.is_dir():
        shutil.rmtree(p)

for name in uploaded.keys():
    src = Path("/content") / name
    dst = INPUT_DIR / name
    shutil.copy(str(src), str(dst))

print("\nUploaded files copied to:")
for p in sorted(INPUT_DIR.glob("*")):
    print("-", p)

Upload current Step4T output files or zip.


Saving Step4_expB_settingA1_implicit_transitive_lr.zip to Step4_expB_settingA1_implicit_transitive_lr.zip

Uploaded files copied to:
- /content/pilot4_synthetic_step4_lr_5000/input_step4/Step4_expB_settingA1_implicit_transitive_lr.zip


In [3]:
# ============================================================
# Load current Step4T output samples
# ============================================================

def load_json_or_jsonl(path: Path) -> List[Dict[str, Any]]:
    if path.suffix == ".jsonl":
        rows = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        return rows

    if path.suffix == ".json":
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        if isinstance(data, list):
            return data

        # Some outputs may wrap records.
        for key in ["records", "samples", "data"]:
            if isinstance(data, dict) and key in data and isinstance(data[key], list):
                return data[key]

        raise ValueError(f"Unsupported JSON structure in {path}")

    raise ValueError(f"Unsupported file type: {path}")


def extract_all_input_files(input_dir: Path) -> List[Path]:
    extracted_dir = input_dir / "_extracted"
    extracted_dir.mkdir(exist_ok=True)

    for zip_path in input_dir.glob("*.zip"):
        with zipfile.ZipFile(str(zip_path), "r") as zf:
            zf.extractall(str(extracted_dir))

    files_found = []
    files_found.extend(input_dir.rglob("*.json"))
    files_found.extend(input_dir.rglob("*.jsonl"))

    # Exclude files inside output if rerun.
    files_found = [
        p for p in files_found
        if OUTPUT_DIR not in p.parents
    ]

    return sorted(set(files_found))


input_files = extract_all_input_files(INPUT_DIR)

if not input_files:
    raise FileNotFoundError("No .json or .jsonl files found in uploaded Step4T output.")

print("Input files found:")
for p in input_files:
    print("-", p.relative_to(INPUT_DIR))

seed_rows = []

for p in input_files:
    try:
        rows = load_json_or_jsonl(p)
        seed_rows.extend(rows)
        print(f"Loaded {len(rows)} rows from {p.name}")
    except Exception as e:
        print(f"Skipped {p}: {e}")

if not seed_rows:
    raise ValueError("No Step4T rows could be loaded.")

print("\nTotal seed rows loaded:", len(seed_rows))


# ============================================================
# Extract object type and uid seeds
# ============================================================

def collect_object_types_and_uids(rows: List[Dict[str, Any]]) -> Tuple[List[str], List[str]]:
    object_types = set()
    object_uids = set()

    for row in rows:
        pp = row.get("probe_pair", {})
        ev = row.get("evidence", {})

        for key in [
            "probe_subject_type",
            "probe_object_type",
        ]:
            if pp.get(key):
                object_types.add(str(pp[key]))

        for key in [
            "probe_subject_uid",
            "probe_object_uid",
            "probe_subject_mention_text",
            "probe_object_mention_text",
        ]:
            if pp.get(key):
                object_uids.add(str(pp[key]))

        for sr in ev.get("supporting_relations", []) or []:
            for key in ["subject_type", "object_type"]:
                if sr.get(key):
                    object_types.add(str(sr[key]))

            for key in ["subject_uid", "object_uid"]:
                if sr.get(key):
                    object_uids.add(str(sr[key]))

    return sorted(object_types), sorted(object_uids)


seed_types, seed_uids = collect_object_types_and_uids(seed_rows)

fallback_types = [
    "Apple", "Mug", "Plate", "Bowl", "Cup", "Fork", "Spoon", "Knife",
    "Book", "Bottle", "Box", "Pan", "Pot", "Kettle", "Lettuce", "Tomato",
    "Potato", "SoapBottle", "WineBottle", "PepperShaker", "SaltShaker",
    "Spatula", "Cabinet", "CounterTop", "DishSponge", "CoffeeMachine",
]

if len(seed_types) < 10:
    object_types = fallback_types
else:
    object_types = seed_types

print("\nObject types used for synthetic generation:")
print(object_types)

print("\nNumber of seed object uids:", len(seed_uids))

Input files found:
- _extracted/p4_expB_settingA1_lr/p4_expB_settingA1_implicit_transitive_lr_spans.json
- _extracted/p4_expB_settingA1_lr/p4_expB_settingA1_lr_output.json
- _extracted/p4_expB_settingA1_lr/p4_expB_settingA1_lr_output_summary.json
- _extracted/p4_expB_settingA1_lr/step4T_FloorPlan1_implicit_transitive_probe_samples.json
- _extracted/p4_expB_settingA1_lr/step4T_FloorPlan2_implicit_transitive_probe_samples.json
- _extracted/p4_expB_settingA1_lr/step4T_FloorPlan3_implicit_transitive_probe_samples.json
- _extracted/p4_expB_settingA1_lr/step4T_FloorPlan4_implicit_transitive_probe_samples.json
- _extracted/p4_expB_settingA1_lr/step4T_FloorPlan5_implicit_transitive_probe_samples.json
- _extracted/p4_expB_settingA1_lr/step4T_FloorPlan6_implicit_transitive_probe_samples.json
Loaded 167 rows from p4_expB_settingA1_implicit_transitive_lr_spans.json
Loaded 167 rows from p4_expB_settingA1_lr_output.json
Skipped /content/pilot4_synthetic_step4_lr_5000/input_step4/_extracted/p4_expB_s

In [4]:
# ============================================================
# Surface templates for spatial relations
#
# Each template realizes a logical relation:
#   subject relation object
#
# But some templates use inverse surface order:
#   object inverse(relation) subject
#
# This avoids binding label to one fixed surface pattern.
# ============================================================

DIRECT_SURFACE_TEMPLATES = {
    "left_of": [
        "{subj} is to the left of {obj}.",
        "{subj} is on the left side of {obj}.",
        "{subj} sits to the left of {obj}.",
        "{subj} is positioned to the left of {obj}.",
        "{subj} can be seen to the left of {obj}.",
        "To the left of {obj}, there is {subj}.",
        "On the left side of {obj}, you can see {subj}.",
    ],
    "right_of": [
        "{subj} is to the right of {obj}.",
        "{subj} is on the right side of {obj}.",
        "{subj} sits to the right of {obj}.",
        "{subj} is positioned to the right of {obj}.",
        "{subj} can be seen to the right of {obj}.",
        "To the right of {obj}, there is {subj}.",
        "On the right side of {obj}, you can see {subj}.",
    ],
}

# For logical relation subj R obj, inverse surface says:
#   obj inverse(R) subj
INVERSE_SURFACE_TEMPLATES = {
    "left_of": [
        "{obj} is to the right of {subj}.",
        "{obj} is on the right side of {subj}.",
        "{obj} sits to the right of {subj}.",
        "{obj} is positioned to the right of {subj}.",
        "To the right of {subj}, there is {obj}.",
        "On the right side of {subj}, you can see {obj}.",
    ],
    "right_of": [
        "{obj} is to the left of {subj}.",
        "{obj} is on the left side of {subj}.",
        "{obj} sits to the left of {subj}.",
        "{obj} is positioned to the left of {subj}.",
        "To the left of {subj}, there is {obj}.",
        "On the left side of {subj}, you can see {obj}.",
    ],
}

DISTRACTOR_TEMPLATES = [
    "{x} is near {y}.",
    "{x} is close to {y}.",
    "{x} appears near {y}.",
    "{x} can be seen close to {y}.",
    "{x} is in the same local area as {y}.",
    "Nearby, {x} and {y} are both visible.",
    "In this part of the scene, {x} appears together with {y}.",
]

INTRO_TEMPLATES = [
    "In this local scene,",
    "In this part of the room,",
    "Within the described area,",
    "In the observed region,",
    "Looking at this local arrangement,",
    "In this synthetic spatial description,",
]


def realize_relation_sentence(
    subj: str,
    relation: str,
    obj: str,
    force_surface_mode: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Realize logical relation:
        subj relation obj

    surface_mode:
        direct:
            subj R obj
        inverse:
            obj inverse(R) subj

    Returns sentence + metadata.
    """
    if force_surface_mode is None:
        surface_mode = random.choice(["direct", "inverse"])
    else:
        surface_mode = force_surface_mode

    if surface_mode == "direct":
        template = random.choice(DIRECT_SURFACE_TEMPLATES[relation])
        sentence = template.format(subj=subj, obj=obj)
        was_swapped_for_text = False
        surface_subject_uid = subj
        surface_object_uid = obj
        surface_relation = relation

    elif surface_mode == "inverse":
        template = random.choice(INVERSE_SURFACE_TEMPLATES[relation])
        sentence = template.format(subj=subj, obj=obj)
        was_swapped_for_text = True
        surface_subject_uid = obj
        surface_object_uid = subj
        surface_relation = INVERSE_RELATION_MAP[relation]

    else:
        raise ValueError(f"Unknown surface_mode: {surface_mode}")

    return {
        "sentence": sentence,
        "template": template,
        "surface_mode": surface_mode,
        "was_swapped_for_text": was_swapped_for_text,
        "surface_subject_uid": surface_subject_uid,
        "surface_object_uid": surface_object_uid,
        "surface_relation": surface_relation,
    }


print("Template library loaded.")

Template library loaded.


In [11]:
# ============================================================
# Build disjoint train/test object pools
# ============================================================

def normalize_type_to_alias_prefix(type_name: str) -> str:
    # CamelCase / mixed type -> lower snake-ish prefix
    s = re.sub(r"[^A-Za-z0-9]+", "_", type_name).strip("_")
    s = re.sub(r"(?<!^)(?=[A-Z])", "_", s).lower()
    s = re.sub(r"_+", "_", s)
    return s or "object"


def build_object_pool(
    pool_name: str,
    object_types: List[str],
    num_objects: int,
) -> List[Dict[str, str]]:
    objects = []

    if pool_name == TRAIN_POOL_NAME:
        pool_offset = 0
    elif pool_name == TEST_POOL_NAME:
        pool_offset = 100000
    else:
        raise ValueError(f"Unknown pool_name: {pool_name}")

    for i in range(num_objects):
        type_name = object_types[i % len(object_types)]
        prefix = normalize_type_to_alias_prefix(type_name)

        # No train_/test_ prefix in uid or mention.
        # This uid is what appears in text.
        visible_index = pool_offset + i
        uid = f"{prefix}_{visible_index:06d}"

        # Keep split marker only in metadata/id, not in text.
        obj_id = f"Synthetic{type_name}|{pool_name}|{i:04d}"

        objects.append({
            "uid": uid,
            "mention": uid,
            "type": type_name,
            "id": obj_id,
            "pool": pool_name,
        })

    random.shuffle(objects)
    return objects


# Need enough objects so samples do not repeatedly use the same small object set.
# 1800 per train pool and 900 per test pool are enough for 5000 samples.
TRAIN_OBJECTS = build_object_pool(TRAIN_POOL_NAME, object_types, num_objects=1800)
TEST_OBJECTS = build_object_pool(TEST_POOL_NAME, object_types, num_objects=900)

TRAIN_UIDS = {o["uid"] for o in TRAIN_OBJECTS}
TEST_UIDS = {o["uid"] for o in TEST_OBJECTS}

assert TRAIN_UIDS.isdisjoint(TEST_UIDS)

print("Train object pool size:", len(TRAIN_OBJECTS))
print("Test object pool size:", len(TEST_OBJECTS))
print("Train/test object uid overlap:", len(TRAIN_UIDS & TEST_UIDS))
print("\nExample train objects:", TRAIN_OBJECTS[:5])
print("\nExample test objects:", TEST_OBJECTS[:5])

Train object pool size: 1800
Test object pool size: 900
Train/test object uid overlap: 0

Example train objects: [{'uid': 'knife_000424', 'mention': 'knife_000424', 'type': 'Knife', 'id': 'SyntheticKnife|train|0424', 'pool': 'train'}, {'uid': 'credit_card_001044', 'mention': 'credit_card_001044', 'type': 'CreditCard', 'id': 'SyntheticCreditCard|train|1044', 'pool': 'train'}, {'uid': 'pan_000249', 'mention': 'pan_000249', 'type': 'Pan', 'id': 'SyntheticPan|train|0249', 'pool': 'train'}, {'uid': 'cup_000460', 'mention': 'cup_000460', 'type': 'Cup', 'id': 'SyntheticCup|train|0460', 'pool': 'train'}, {'uid': 'book_000631', 'mention': 'book_000631', 'type': 'Book', 'id': 'SyntheticBook|train|0631', 'pool': 'train'}]

Example test objects: [{'uid': 'shelf_100256', 'mention': 'shelf_100256', 'type': 'Shelf', 'id': 'SyntheticShelf|test|0256', 'pool': 'test'}, {'uid': 'kettle_100783', 'mention': 'kettle_100783', 'type': 'Kettle', 'id': 'SyntheticKettle|test|0783', 'pool': 'test'}, {'uid': 'wind

In [12]:
# ============================================================
# Core synthetic sample generation
# ============================================================

def find_all_char_spans(text: str, mention: str) -> List[Dict[str, int]]:
    spans = []
    start = 0

    while True:
        idx = text.find(mention, start)
        if idx == -1:
            break

        spans.append({
            "start": idx,
            "end": idx + len(mention),
        })
        start = idx + len(mention)

    return spans


def choose_three_distinct_objects(pool: List[Dict[str, str]]) -> Tuple[Dict[str, str], Dict[str, str], Dict[str, str]]:
    objs = random.sample(pool, 3)
    return objs[0], objs[1], objs[2]


def choose_distractor_objects(pool: List[Dict[str, str]], exclude_uids: set, k: int = 2) -> List[Dict[str, str]]:
    candidates = [o for o in pool if o["uid"] not in exclude_uids]
    if len(candidates) < k:
        return []
    return random.sample(candidates, k)


def build_support_edges(
    A: Dict[str, str],
    B: Dict[str, str],
    C: Dict[str, str],
    target_relation: str,
    implicit_rule: str,
) -> List[Tuple[Dict[str, str], str, Dict[str, str]]]:
    """
    Build semantic support edges for target:
        A target_relation C
    """
    inv = INVERSE_RELATION_MAP[target_relation]

    if implicit_rule == "chain_same_direction":
        # A R B, B R C => A R C
        return [
            (A, target_relation, B),
            (B, target_relation, C),
        ]

    if implicit_rule == "shared_anchor_opposite_sides":
        # A R B, C inv(R) B => A R C
        return [
            (A, target_relation, B),
            (C, inv, B),
        ]

    if implicit_rule == "anchor_between_reversed_surface_form":
        # B inv(R) A, B R C => A R C
        return [
            (B, inv, A),
            (B, target_relation, C),
        ]

    raise ValueError(f"Unknown implicit_rule: {implicit_rule}")


def make_supporting_relation_record(
    subj: Dict[str, str],
    relation: str,
    obj: Dict[str, str],
    sentence_info: Dict[str, Any],
    generation_mode: str = "synthetic",
    template_index: int = -1,
) -> Dict[str, Any]:
    return {
        "sentence": sentence_info["sentence"],
        "template": sentence_info["template"],
        "template_index": template_index,
        "generation_mode": generation_mode,

        # Logical relation metadata.
        "subject_uid": subj["uid"],
        "subject_id": subj["id"],
        "subject_type": subj["type"],
        "relation": relation,
        "object_uid": obj["uid"],
        "object_id": obj["id"],
        "object_type": obj["type"],

        # Surface realization metadata.
        "was_swapped_for_text": sentence_info["was_swapped_for_text"],
        "surface_mode": sentence_info["surface_mode"],
        "surface_subject_uid": sentence_info["surface_subject_uid"],
        "surface_relation": sentence_info["surface_relation"],
        "surface_object_uid": sentence_info["surface_object_uid"],

        "source_relation": {
            "subject_id": subj["id"],
            "relation": relation,
            "object_id": obj["id"],
            "relation_family": "synthetic_geometric",
        },
    }


def build_synthetic_sample(
    sample_index: int,
    scene: str,
    object_pool: List[Dict[str, str]],
    target_relation: str,
    implicit_rule: str,
) -> Dict[str, Any]:
    """
    Build one Step4T-style implicit-transitive sample.
    """

    A, B, C = choose_three_distinct_objects(object_pool)

    # Target is A target_relation C.
    support_edges = build_support_edges(A, B, C, target_relation, implicit_rule)

    supporting_relations = []
    support_sentences = []

    # Balance surface order by random direct/inverse realization.
    # This avoids label-template one-to-one mapping.
    for edge_idx, (subj, rel, obj) in enumerate(support_edges):
        sentence_info = realize_relation_sentence(
            subj=subj["uid"],
            relation=rel,
            obj=obj["uid"],
            force_surface_mode=None,
        )

        supporting_relations.append(
            make_supporting_relation_record(
                subj=subj,
                relation=rel,
                obj=obj,
                sentence_info=sentence_info,
                template_index=edge_idx,
            )
        )

        support_sentences.append(sentence_info["sentence"])

    # Add distractor sentences that do not state target A-R-C.
    exclude = {A["uid"], B["uid"], C["uid"]}
    distractor_sentences = []

    num_distractors = random.choice([2, 3, 4])

    for _ in range(num_distractors):
        d_objs = choose_distractor_objects(object_pool, exclude_uids=exclude, k=2)
        if len(d_objs) < 2:
            continue

        x, y = d_objs
        template = random.choice(DISTRACTOR_TEMPLATES)
        distractor_sentences.append(template.format(x=x["uid"], y=y["uid"]))

    intro = random.choice(INTRO_TEMPLATES)

    # Shuffle support and distractor sentences, but keep all supports present.
    all_sentences = support_sentences + distractor_sentences
    random.shuffle(all_sentences)

    text = intro + " " + " ".join(all_sentences)

    # Sanity check: target direct sentence should not be explicitly present.
    # This is a conservative string-level check, not exhaustive semantic parsing.
    target_direct_forms = [
        f"{A['uid']} is to the left of {C['uid']}.",
        f"{A['uid']} is to the right of {C['uid']}.",
        f"{C['uid']} is to the left of {A['uid']}.",
        f"{C['uid']} is to the right of {A['uid']}.",
    ]

    for forbidden in target_direct_forms:
        if forbidden in text:
            raise ValueError(f"Target relation leaked into text: {forbidden}")

    subject_spans = find_all_char_spans(text, A["uid"])
    object_spans = find_all_char_spans(text, C["uid"])

    if not subject_spans or not object_spans:
        raise ValueError(
            f"Failed to locate probe subject/object spans. "
            f"A={A['uid']}, C={C['uid']}, text={text}"
        )

    sample_id = f"synthetic_step4T_{sample_index:06d}_{uuid.uuid4().hex[:8]}"

    paragraph_id = f"synthetic_{scene}_paragraph_{sample_index:06d}"
    chunk_id = f"synthetic_{scene}_chunk_{sample_index // 1000}"
    cluster_id = f"{chunk_id}_cluster_{sample_index % 10}"

    pair_group_id = (
        f"{paragraph_id}|{A['uid']}|{C['uid']}|"
        f"{target_relation}|implicit_transitive|{implicit_rule}"
    )

    row = {
        "sample_id": sample_id,
        "pair_group_id": pair_group_id,

        "scene": scene,
        "experiment": "Step4_expB_settingA1_implicit_transitive_lr_synthetic_5000",
        "generation_mode": "synthetic",
        "paragraph_id": paragraph_id,
        "chunk_id": chunk_id,
        "cluster_id": cluster_id,
        "grid_i": None,
        "grid_j": None,
        "paragraph_index_within_cluster": sample_index,

        "text": text,

        "evidence": {
            "evidence_type": "implicit_transitive",
            "probe_direction_relative_to_text": "implicit",
            "has_explicit_evidence_sentence": False,
            "evidence_sentence": None,
            "sentence_index_in_paragraph": None,
            "surface_order": None,

            "supporting_relation_source": "synthetic_explicit_relations",
            "implicit_rule": implicit_rule,
            "anchor_uid": B["uid"],
            "supporting_relations": supporting_relations,
            "num_supporting_paths": 1,
            "original_pair_evidence_type": "synthetic_implicit_labeled",
            "label_source": "synthetic_controlled_transitive_generation",

            "all_supporting_paths": [
                {
                    "implicit_rule": implicit_rule,
                    "anchor_uid": B["uid"],
                    "supporting_relations": supporting_relations,
                }
            ],
        },

        "probe_pair": {
            "probe_subject_uid": A["uid"],
            "probe_subject_id": A["id"],
            "probe_subject_type": A["type"],
            "probe_subject_mention_text": A["uid"],
            "probe_subject_all_char_spans_in_paragraph": subject_spans,
            "probe_subject_span_in_evidence_sentence": None,
            "probe_subject_span_in_paragraph": subject_spans[0],

            "probe_object_uid": C["uid"],
            "probe_object_id": C["id"],
            "probe_object_type": C["type"],
            "probe_object_mention_text": C["uid"],
            "probe_object_all_char_spans_in_paragraph": object_spans,
            "probe_object_span_in_evidence_sentence": None,
            "probe_object_span_in_paragraph": object_spans[0],

            "probe_relation_label": target_relation,
            "label_space": "directed_spatial_relation",
            "has_relation_label": True,
            "is_explicit_in_text": False,
            "is_inverse_of_text_relation": False,
            "is_symmetric_relation": False,
            "is_directional_relation": True,
            "is_implicit_transitive": True,
        },

        "source_relation": {
            "relation_source": "synthetic_transitive_rule",
            "source_relation_summary": {
                "subject_id": A["id"],
                "relation": target_relation,
                "object_id": C["id"],
                "relation_family": "synthetic_geometric",
            },
            "label_source": "synthetic_controlled_transitive_generation",
            "sample_source": "synthetic_step4T_generator",
            "text_source": "synthetic_paragraph_text",
            "original_pair_target": {
                "subject_uid": A["uid"],
                "subject_id": A["id"],
                "subject_type": A["type"],
                "relation_label": target_relation,
                "object_uid": C["uid"],
                "object_id": C["id"],
                "object_type": C["type"],
                "pair_evidence_type": "synthetic_implicit_labeled",
                "relation_source": "synthetic_direct",
            },
        },

        "geometry": {
            "has_geometry_label": False,
            "geometry_label": None,
        },

        "source_step3_file": None,
        "source_step3_scene": scene,

        # Synthetic diagnostics.
        "synthetic_metadata": {
            "is_synthetic": True,
            "target_num_samples": TARGET_NUM_SAMPLES,
            "object_pool": A["pool"],
            "target_relation": target_relation,
            "implicit_rule": implicit_rule,
            "template_control": {
                "uses_direct_and_inverse_surface_forms": True,
                "target_relation_not_explicitly_written": True,
                "train_test_object_names_disjoint": True,
            },
        },
    }

    return row


print("Synthetic sample generator loaded.")

Synthetic sample generator loaded.


In [13]:
# ============================================================
# Generate 5000 synthetic Step4T-style samples
#
# Important:
#   Balance label and implicit_rule within each scene.
#   Avoid binding label to scene.
# ============================================================

NUM_TRAIN_SYNTHETIC = int(TARGET_NUM_SAMPLES * 0.8)
NUM_TEST_SYNTHETIC = TARGET_NUM_SAMPLES - NUM_TRAIN_SYNTHETIC

print("NUM_TRAIN_SYNTHETIC:", NUM_TRAIN_SYNTHETIC)
print("NUM_TEST_SYNTHETIC:", NUM_TEST_SYNTHETIC)


def build_balanced_schedule(
    num_samples: int,
    scenes: List[str],
    object_pool: List[Dict[str, str]],
    split_name: str,
) -> List[Dict[str, Any]]:
    """
    Build a near-balanced schedule over:
        scene x label x implicit_rule

    This avoids label-scene binding.
    """
    combos = []

    for scene in scenes:
        for label in LABELS:
            for rule in IMPLICIT_RULES:
                combos.append({
                    "split": split_name,
                    "scene": scene,
                    "label": label,
                    "implicit_rule": rule,
                    "pool": object_pool,
                })

    schedule = []

    full_rounds = num_samples // len(combos)
    remainder = num_samples % len(combos)

    for _ in range(full_rounds):
        schedule.extend(combos)

    # Add remainder using shuffled combos to avoid deterministic bias.
    extra = combos[:]
    random.shuffle(extra)
    schedule.extend(extra[:remainder])

    random.shuffle(schedule)
    return schedule


train_schedule = build_balanced_schedule(
    num_samples=NUM_TRAIN_SYNTHETIC,
    scenes=TRAIN_SCENES,
    object_pool=TRAIN_OBJECTS,
    split_name="train",
)

test_schedule = build_balanced_schedule(
    num_samples=NUM_TEST_SYNTHETIC,
    scenes=TEST_SCENES,
    object_pool=TEST_OBJECTS,
    split_name="test",
)

schedule = train_schedule + test_schedule
random.shuffle(schedule)

synthetic_rows = []

for i, item in enumerate(schedule):
    row = build_synthetic_sample(
        sample_index=i,
        scene=item["scene"],
        object_pool=item["pool"],
        target_relation=item["label"],
        implicit_rule=item["implicit_rule"],
    )
    synthetic_rows.append(row)

print("Generated synthetic rows:", len(synthetic_rows))

# Basic distribution checks.
print("\nLabel counts:")
print(Counter(r["probe_pair"]["probe_relation_label"] for r in synthetic_rows))

print("\nImplicit rule counts:")
print(Counter(r["evidence"]["implicit_rule"] for r in synthetic_rows))

print("\nScene counts:")
print(Counter(r["scene"] for r in synthetic_rows))

print("\nScene x label counts:")
print(Counter((r["scene"], r["probe_pair"]["probe_relation_label"]) for r in synthetic_rows))

print("\nScene x implicit_rule counts:")
print(Counter((r["scene"], r["evidence"]["implicit_rule"]) for r in synthetic_rows))

print("\nObject pool counts:")
print(Counter(r["synthetic_metadata"]["object_pool"] for r in synthetic_rows))

NUM_TRAIN_SYNTHETIC: 4000
NUM_TEST_SYNTHETIC: 1000
Generated synthetic rows: 5000

Label counts:
Counter({'right_of': 2501, 'left_of': 2499})

Implicit rule counts:
Counter({'shared_anchor_opposite_sides': 1667, 'anchor_between_reversed_surface_form': 1667, 'chain_same_direction': 1666})

Scene counts:
Counter({'FloorPlan4': 1001, 'FloorPlan3': 1000, 'FloorPlan1': 1000, 'FloorPlan2': 999, 'FloorPlan5': 501, 'FloorPlan6': 499})

Scene x label counts:
Counter({('FloorPlan4', 'right_of'): 501, ('FloorPlan2', 'right_of'): 500, ('FloorPlan3', 'right_of'): 500, ('FloorPlan3', 'left_of'): 500, ('FloorPlan4', 'left_of'): 500, ('FloorPlan1', 'left_of'): 500, ('FloorPlan1', 'right_of'): 500, ('FloorPlan2', 'left_of'): 499, ('FloorPlan5', 'left_of'): 251, ('FloorPlan6', 'right_of'): 250, ('FloorPlan5', 'right_of'): 250, ('FloorPlan6', 'left_of'): 249})

Scene x implicit_rule counts:
Counter({('FloorPlan4', 'chain_same_direction'): 334, ('FloorPlan3', 'shared_anchor_opposite_sides'): 334, ('FloorP

In [14]:
# ============================================================
# Quality checks
# ============================================================

def check_target_not_explicit(row: Dict[str, Any]) -> bool:
    text = row["text"]
    pp = row["probe_pair"]

    A = pp["probe_subject_uid"]
    C = pp["probe_object_uid"]

    forbidden_patterns = [
        f"{A} is to the left of {C}",
        f"{A} is to the right of {C}",
        f"{C} is to the left of {A}",
        f"{C} is to the right of {A}",
        f"To the left of {C}, there is {A}",
        f"To the right of {C}, there is {A}",
        f"To the left of {A}, there is {C}",
        f"To the right of {A}, there is {C}",
    ]

    return not any(p in text for p in forbidden_patterns)


def check_spans(row: Dict[str, Any]) -> bool:
    text = row["text"]
    pp = row["probe_pair"]

    for key, mention_key in [
        ("probe_subject_span_in_paragraph", "probe_subject_mention_text"),
        ("probe_object_span_in_paragraph", "probe_object_mention_text"),
    ]:
        span = pp[key]
        mention = pp[mention_key]

        if span is None:
            return False

        start, end = span["start"], span["end"]

        if start < 0 or end <= start or end > len(text):
            return False

        if text[start:end] != mention:
            return False

    return True


# Check target leakage.
leaked = [r["sample_id"] for r in synthetic_rows if not check_target_not_explicit(r)]
bad_spans = [r["sample_id"] for r in synthetic_rows if not check_spans(r)]

print("Target leakage count:", len(leaked))
print("Bad span count:", len(bad_spans))

if leaked[:5]:
    print("Example leaked ids:", leaked[:5])

if bad_spans[:5]:
    print("Example bad span ids:", bad_spans[:5])

if leaked:
    raise ValueError(f"Found {len(leaked)} samples with possible target leakage.")

if bad_spans:
    raise ValueError(f"Found {len(bad_spans)} samples with bad spans.")


# Check train/test object overlap by scene.
train_used = set()
test_used = set()

for row in synthetic_rows:
    scene = row["scene"]
    pp = row["probe_pair"]

    uids = {
        pp["probe_subject_uid"],
        pp["probe_object_uid"],
        row["evidence"]["anchor_uid"],
    }

    for sr in row["evidence"]["supporting_relations"]:
        uids.add(sr["subject_uid"])
        uids.add(sr["object_uid"])

    if scene in TRAIN_SCENES:
        train_used.update(uids)
    elif scene in TEST_SCENES:
        test_used.update(uids)

overlap = train_used & test_used

print("\nTrain used objects:", len(train_used))
print("Test used objects:", len(test_used))
print("Train/test object overlap:", len(overlap))

if overlap:
    raise ValueError(f"Train/test object overlap detected: {list(overlap)[:10]}")

# Check scene-label balance.
scene_label_counts = Counter(
    (r["scene"], r["probe_pair"]["probe_relation_label"])
    for r in synthetic_rows
)

print("\nScene x label counts:")
for k, v in sorted(scene_label_counts.items()):
    print(k, v)

bad_scene_labels = []

for scene in ALL_SCENES:
    labels_in_scene = {
        label
        for (s, label), count in scene_label_counts.items()
        if s == scene and count > 0
    }

    if set(LABELS) - labels_in_scene:
        bad_scene_labels.append({
            "scene": scene,
            "labels_present": sorted(labels_in_scene),
            "labels_missing": sorted(set(LABELS) - labels_in_scene),
        })

if bad_scene_labels:
    raise ValueError(
        "Some scenes do not contain both left_of and right_of labels: "
        + json.dumps(bad_scene_labels, indent=2, ensure_ascii=False)
    )


# Check surface mode balance.
surface_mode_counts = Counter()
label_surface_counts = Counter()

for row in synthetic_rows:
    label = row["probe_pair"]["probe_relation_label"]
    for sr in row["evidence"]["supporting_relations"]:
        mode = sr.get("surface_mode")
        surface_mode_counts[mode] += 1
        label_surface_counts[(label, mode)] += 1

print("\nSurface mode counts across supporting relations:")
print(surface_mode_counts)

print("\nLabel x surface mode counts:")
for k, v in sorted(label_surface_counts.items()):
    print(k, v)


# Check supporting sentence template variety.
template_counts = Counter()

for row in synthetic_rows:
    for sr in row["evidence"]["supporting_relations"]:
        template_counts[sr["template"]] += 1

print("\nNumber of unique supporting templates used:", len(template_counts))
print("Top 10 templates:")
for tpl, cnt in template_counts.most_common(10):
    print(cnt, "|", tpl)

print("\nQuality checks passed.")

Target leakage count: 0
Bad span count: 0

Train used objects: 1798
Test used objects: 863
Train/test object overlap: 0

Scene x label counts:
('FloorPlan1', 'left_of') 500
('FloorPlan1', 'right_of') 500
('FloorPlan2', 'left_of') 499
('FloorPlan2', 'right_of') 500
('FloorPlan3', 'left_of') 500
('FloorPlan3', 'right_of') 500
('FloorPlan4', 'left_of') 500
('FloorPlan4', 'right_of') 501
('FloorPlan5', 'left_of') 251
('FloorPlan5', 'right_of') 250
('FloorPlan6', 'left_of') 249
('FloorPlan6', 'right_of') 250

Surface mode counts across supporting relations:
Counter({'direct': 5023, 'inverse': 4977})

Label x surface mode counts:
('left_of', 'direct') 2499
('left_of', 'inverse') 2499
('right_of', 'direct') 2524
('right_of', 'inverse') 2478

Number of unique supporting templates used: 26
Top 10 templates:
441 | {obj} is on the right side of {subj}.
439 | To the left of {subj}, there is {obj}.
430 | {obj} is positioned to the right of {subj}.
430 | {obj} sits to the left of {subj}.
429 | {obj}

In [15]:
# ============================================================
# Save synthetic dataset
# ============================================================

output_json_path = OUTPUT_DIR / OUTPUT_JSON
output_step5_compat_json_path = OUTPUT_DIR / STEP5_COMPAT_JSON
output_summary_path = OUTPUT_DIR / f"{OUTPUT_BASENAME}_summary.json"
output_zip_path = Path("/content") / OUTPUT_ZIP

# Main requested filename.
with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(synthetic_rows, f, indent=2, ensure_ascii=False)

# Step5-compatible filename.
# This lets your existing Step5 discover_step4_input_files() find the synthetic dataset
# without changing Step5 input pattern.
with open(output_step5_compat_json_path, "w", encoding="utf-8") as f:
    json.dump(synthetic_rows, f, indent=2, ensure_ascii=False)

summary = {
    "output_file": OUTPUT_JSON,
    "step5_compatible_output_file": STEP5_COMPAT_JSON,
    "num_samples": len(synthetic_rows),
    "random_seed": RANDOM_SEED,
    "target_num_samples": TARGET_NUM_SAMPLES,
    "format": "Step4T-style implicit_transitive probe samples",

    "label_counts": dict(Counter(r["probe_pair"]["probe_relation_label"] for r in synthetic_rows)),
    "implicit_rule_counts": dict(Counter(r["evidence"]["implicit_rule"] for r in synthetic_rows)),
    "scene_counts": dict(Counter(r["scene"] for r in synthetic_rows)),
    "scene_label_counts": {
        f"{scene}|{label}": count
        for (scene, label), count in Counter(
            (r["scene"], r["probe_pair"]["probe_relation_label"])
            for r in synthetic_rows
        ).items()
    },
    "scene_implicit_rule_counts": {
        f"{scene}|{rule}": count
        for (scene, rule), count in Counter(
            (r["scene"], r["evidence"]["implicit_rule"])
            for r in synthetic_rows
        ).items()
    },
    "object_pool_counts": dict(Counter(r["synthetic_metadata"]["object_pool"] for r in synthetic_rows)),

    "train_scenes": TRAIN_SCENES,
    "test_scenes": TEST_SCENES,
    "train_used_object_count": len(train_used),
    "test_used_object_count": len(test_used),
    "train_test_object_overlap": len(overlap),

    "surface_mode_counts": dict(surface_mode_counts),
    "label_surface_counts": {
        f"{label}|{mode}": count
        for (label, mode), count in label_surface_counts.items()
    },
    "num_unique_supporting_templates": len(template_counts),

    "design_controls": {
        "target_relation_not_explicitly_written": True,
        "left_right_balanced": True,
        "implicit_rule_balanced": True,
        "scene_label_balanced": True,
        "direct_inverse_surface_forms_mixed": True,
        "train_test_object_names_disjoint": True,
        "main_output_filename_preserved": True,
        "step5_compatible_copy_saved": True,
    },
}

with open(output_summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

# Zip output files.
if output_zip_path.exists():
    output_zip_path.unlink()

with zipfile.ZipFile(str(output_zip_path), "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(str(output_json_path), arcname=OUTPUT_JSON)
    zf.write(str(output_step5_compat_json_path), arcname=STEP5_COMPAT_JSON)
    zf.write(str(output_summary_path), arcname=f"{OUTPUT_BASENAME}_summary.json")

print("Saved synthetic dataset:")
print(output_json_path)

print("\nSaved Step5-compatible copy:")
print(output_step5_compat_json_path)

print("\nSaved summary:")
print(output_summary_path)

print("\nCreated zip:")
print(output_zip_path)

print("\nSummary:")
print(json.dumps(summary, indent=2, ensure_ascii=False))

Saved synthetic dataset:
/content/pilot4_synthetic_step4_lr_5000/output/pilot4_synthetic_step4_lr_5000.json

Saved Step5-compatible copy:
/content/pilot4_synthetic_step4_lr_5000/output/step4T_pilot4_synthetic_step4_lr_5000_implicit_transitive_probe_samples.json

Saved summary:
/content/pilot4_synthetic_step4_lr_5000/output/pilot4_synthetic_step4_lr_5000_summary.json

Created zip:
/content/pilot4_synthetic_step4_lr_5000.zip

Summary:
{
  "output_file": "pilot4_synthetic_step4_lr_5000.json",
  "step5_compatible_output_file": "step4T_pilot4_synthetic_step4_lr_5000_implicit_transitive_probe_samples.json",
  "num_samples": 5000,
  "random_seed": 42,
  "target_num_samples": 5000,
  "format": "Step4T-style implicit_transitive probe samples",
  "label_counts": {
    "right_of": 2501,
    "left_of": 2499
  },
  "implicit_rule_counts": {
    "shared_anchor_opposite_sides": 1667,
    "chain_same_direction": 1666,
    "anchor_between_reversed_surface_form": 1667
  },
  "scene_counts": {
    "Floor

In [16]:
# ============================================================
# Download synthetic dataset zip
# ============================================================

files.download(str(output_zip_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>